'''
    @Author:Ankitha
    @Date: 02-01-2025
    @Last Modified by: Ankitha
    @Last Modified time: 02-01-2025
    @Title :PySpark Sql programs
'''

In [4]:
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.appName("Covid_analysis").getOrCreate()

input_file = "owid-covid-data.csv"
df = spark.read.csv(input_file,header=True,inferSchema=True)

### Q1.Filter Data for a Specific Date

In [10]:
filtered_data = df.filter(df.date == '2021-2-5').select('iso_code','iso_code','location')
filtered_data.show(5)

+--------+--------+--------------+
|iso_code|iso_code|      location|
+--------+--------+--------------+
|     AFG|     AFG|   Afghanistan|
|OWID_AFR|OWID_AFR|        Africa|
|     ALB|     ALB|       Albania|
|     DZA|     DZA|       Algeria|
|     ASM|     ASM|American Samoa|
+--------+--------+--------------+
only showing top 5 rows



### Q2.Select Relevant Columns

In [7]:
selected_data=df.select('iso_code','continent','location')

selected_data.show(3)

+--------+---------+-----------+
|iso_code|continent|   location|
+--------+---------+-----------+
|     AFG|     Asia|Afghanistan|
|     AFG|     Asia|Afghanistan|
|     AFG|     Asia|Afghanistan|
+--------+---------+-----------+
only showing top 3 rows



### Q3.Sort the Data by total_cases

In [8]:
sorted_data = selected_data.orderBy(selected_data.continent.desc())

sorted_data.show(4)

+--------+-------------+---------+
|iso_code|    continent| location|
+--------+-------------+---------+
|     URY|South America|  Uruguay|
|     ARG|South America|Argentina|
|     BOL|South America|  Bolivia|
|     ARG|South America|Argentina|
+--------+-------------+---------+
only showing top 4 rows



In [15]:
df.createOrReplaceTempView("covid")

### Q4.Count the total number of records in the dataset

In [11]:
result = spark.sql("select count(*) as total_records from covid")
result.show()

+-------------+
|total_records|
+-------------+
|       380614|
+-------------+



### Q5.Find the total number of cases reported globally on a specific date.

In [12]:
result = spark.sql("select sum(new_cases) as total_newcases from covid")
result.show()

+--------------+
|total_newcases|
+--------------+
| 3.283668361E9|
+--------------+



### Q6.Retrieve records for a specific country (e.g., India) over a given date range.

In [17]:
result = spark.sql("""
    SELECT iso_code, continent, location, new_cases, population_density 
    FROM covid 
    WHERE continent = 'South America' 
    AND date BETWEEN '2020-01-05' AND '2020-01-08'
""")
result.show(5)

+--------+-------------+---------+---------+------------------+
|iso_code|    continent| location|new_cases|population_density|
+--------+-------------+---------+---------+------------------+
|     ARG|South America|Argentina|      0.0|            16.177|
|     ARG|South America|Argentina|      0.0|            16.177|
|     ARG|South America|Argentina|      0.0|            16.177|
|     ARG|South America|Argentina|      0.0|            16.177|
|     BOL|South America|  Bolivia|      0.0|            10.202|
+--------+-------------+---------+---------+------------------+
only showing top 5 rows



### Q7.Find the top 5 countries with the highest number of new cases on 2021-12-31.

In [17]:
result = spark.sql("select location from covid where date = '2020-1-8' order by new_cases desc limit 5")
result.show()

+----------+
|  location|
+----------+
|Uzbekistan|
|     Nauru|
|  Paraguay|
|     Nepal|
|Madagascar|
+----------+



### Q8.Calculate the total number of new cases and deaths for each country

In [18]:
result = spark.sql("select sum(new_cases),sum(total_deaths) from covid group by location")
result.show()

+--------------+-----------------+
|sum(new_cases)|sum(total_deaths)|
+--------------+-----------------+
|        3904.0|           8755.0|
|      231990.0|        8139343.0|
|   1.3140355E7|     2.63426626E8|
|      272010.0|        7253260.0|
|   1.0084295E7|      1.3940181E8|
|      107325.0|        1894072.0|
|     4859002.0|      3.7521964E7|
|      334863.0|        3714543.0|
|       38084.0|         785509.0|
|      994037.0|        6789551.0|
|        8359.0|          22497.0|
|       48015.0|         181116.0|
|     2047887.0|      3.0203763E7|
|      108047.0|         436056.0|
|       44224.0|         246811.0|
|      834785.0|        9774898.0|
|      451426.0|        8707381.0|
|        9106.0|         130304.0|
|  3.01411932E8|     1.51542766E9|
|   1.1785451E7|      1.3810085E7|
+--------------+-----------------+
only showing top 20 rows



### Q9.Determine the first date when each country reported more than 100,000 total cases.

In [20]:
result = spark.sql("select min(date),location from covid where total_cases > 500 group by location" )
result.show()

+----------+-------------------+
| min(date)|           location|
+----------+-------------------+
|2021-10-10|           Anguilla|
|2020-04-12|        Afghanistan|
|2020-03-22|             Africa|
|2020-04-05|            Algeria|
|2020-03-22|          Argentina|
|2020-07-19|             Angola|
|2020-03-15|            Belgium|
|2020-04-19|            Albania|
|2020-08-02|            Bahamas|
|2020-04-12|            Belarus|
|2022-03-13|     American Samoa|
|2020-04-12|            Andorra|
|2020-04-12|         Bangladesh|
|2021-01-10|           Barbados|
|2020-08-16|              Aruba|
|2020-04-12|         Azerbaijan|
|2020-04-05|            Armenia|
|2021-02-21|Antigua and Barbuda|
|2020-01-26|               Asia|
|2020-03-22|          Australia|
+----------+-------------------+
only showing top 20 rows



### Q10.Find the top 5 countries with the highest life_expectancy

In [30]:
result = spark.sql("SELECT DISTINCT life_expectancy, location FROM covid ORDER BY life_expectancy DESC LIMIT 5")
result.show()


+---------------+----------+
|life_expectancy|  location|
+---------------+----------+
|          86.75|    Monaco|
|          84.97|San Marino|
|          84.86| Hong Kong|
|          84.63|     Japan|
|          84.24|     Macao|
+---------------+----------+

